# Ingestión del archivo `country.json`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo JSON usando `DataFrameReader` de Spark

In [0]:
country_schema = "countryId INT, countryIsoCode STRING, countryName STRING"

countries_df = (spark.read 
    .schema(country_schema)
    .json(f"{bronze_folder_path}/{v_file_date}/country.json")
)
display(countries_df)

countryId,countryIsoCode,countryName
128,AE,United Arab Emirates
129,AF,Afghanistan
130,AO,Angola
131,AR,Argentina
132,AT,Austria
133,AU,Australia
134,AW,Aruba
135,BA,Bosnia and Herzegovina
136,BE,Belgium
137,BG,Bulgaria


## 2. Eliminar las columnas no deseadas del DataFrame

In [0]:
countries_dropped_df = countries_df.drop("countryIsoCode")

## 3. Cambiar el nombre de las columnas según lo requerido

In [0]:
countries_renamed_df = (countries_dropped_df
    .withColumnRenamed("countryId", "country_id")
    .withColumnRenamed("countryName", "country_name")
)

## 4. Agregar las columnas `ingestion_date` y `environmate` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

countries_final_df = add_ingestion_date(countries_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))


## 5. Escribir datos en el datalake en formato `Parquet`

In [0]:
countries_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.countries")

In [0]:
%sql
SELECT * FROM movie_silver.countries

country_id,country_name,ingestion_date,enviroment,file_date
128,United Arab Emirates,2026-09-13T17:51:34.363Z,,2024-12-16
129,Afghanistan,2026-09-13T17:51:34.363Z,,2024-12-16
130,Angola,2026-09-13T17:51:34.363Z,,2024-12-16
131,Argentina,2026-09-13T17:51:34.363Z,,2024-12-16
132,Austria,2026-09-13T17:51:34.363Z,,2024-12-16
133,Australia,2026-09-13T17:51:34.363Z,,2024-12-16
134,Aruba,2026-09-13T17:51:34.363Z,,2024-12-16
135,Bosnia and Herzegovina,2026-09-13T17:51:34.363Z,,2024-12-16
136,Belgium,2026-09-13T17:51:34.363Z,,2024-12-16
137,Bulgaria,2026-09-13T17:51:34.363Z,,2024-12-16
